# 🎬 Netflix Data Cleaning, Auditing & Feature Engineering
### *A Production-Grade Data Analyst ETL Pipeline*
---
**Author**: Data Analytics Team  
**Dataset**: Netflix Movies & TV Shows Dataset (8,800+ Titles)  
**Objective**: Clean raw catalog data, remediate schema anomalies, engineer analytical dimensions, and export structured artifacts for OLAP analysis.



In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)



## 1. Load Raw Dataset & Initial Assessment


In [ ]:
raw_path = Path('../data/raw/netflix_titles.csv')
if not raw_path.exists():
    raw_path = Path('../Netflix-Data-Analysis-main/data/netflix_titles.csv')

df = pd.read_csv(raw_path)
print(f"Dataset Dimension: {df.shape[0]} Rows, {df.shape[1]} Columns\n")
df.info()
df.head(5)



## 2. Missing Value Analysis & Profiling


In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_Percentage': round((df.isna().sum() / len(df)) * 100, 2)
}).sort_values('Missing_Count', ascending=False)

missing_df[missing_df['Missing_Count'] > 0]



## 3. Data Cleaning & Anomaly Remediation
### Anomaly Fix: Ratings containing duration strings (e.g., '74 min', '84 min')
In this dataset, a few entries have durations mistakenly recorded in the `rating` column. We detect and shift these records.



In [ ]:
cleaned = df.copy()

# 1. Clean string column whitespaces
str_cols = ['title', 'director', 'cast', 'country', 'rating', 'duration', 'listed_in', 'description']
for col in str_cols:
    cleaned[col] = cleaned[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# 2. Fix rating-duration misalignment anomaly
duration_in_rating = cleaned['rating'].str.contains('min', na=False)
for idx in cleaned[duration_in_rating].index:
    cleaned.loc[idx, 'duration'] = cleaned.loc[idx, 'rating']
    cleaned.loc[idx, 'rating'] = np.nan

# 3. Impute categorical missing values
cleaned['director_clean'] = cleaned['director'].fillna('Unknown Director')
cleaned['cast_clean'] = cleaned['cast'].fillna('Unknown Cast')
cleaned['country_clean'] = cleaned['country'].fillna('Unknown Country')
cleaned['rating_clean'] = cleaned['rating'].fillna('TV-MA')  # Dominant mode

print("Anomaly remediation completed. Missing ratings count now:", cleaned['rating_clean'].isna().sum())



## 4. Advanced Feature Engineering
We engineer:
- `date_added_dt`: Standard datetime object
- `year_added`, `month_added`, `month_name_added`, `day_name_added`
- `duration_num` & `duration_type`: Clean numeric runtime & units (minutes vs seasons)
- `target_audience` & `age_group`: Demographic segmentation (Kids, Teens, Adults)
- `content_age_at_addition` & `freshness_category`: Catalog licensing gap vs Originals



In [ ]:
# Date Engineering
cleaned['date_added_dt'] = pd.to_datetime(cleaned['date_added'].astype(str).str.strip(), errors='coerce')
missing_dates = cleaned['date_added_dt'].isna()
cleaned.loc[missing_dates, 'date_added_dt'] = pd.to_datetime(
    cleaned.loc[missing_dates, 'release_year'].astype(str) + '-01-01', errors='coerce'
)

cleaned['year_added'] = cleaned['date_added_dt'].dt.year.fillna(cleaned['release_year']).astype(int)
cleaned['month_added'] = cleaned['date_added_dt'].dt.month.fillna(1).astype(int)
cleaned['month_name_added'] = cleaned['date_added_dt'].dt.month_name().fillna('January')
cleaned['day_name_added'] = cleaned['date_added_dt'].dt.day_name().fillna('Monday')

# Duration Parsing
def parse_duration(row):
    dur_str = str(row['duration'])
    if pd.isna(dur_str) or dur_str == 'nan':
        return 0, 'Unknown'
    nums = re.findall(r'\d+', dur_str)
    val = int(nums[0]) if nums else 0
    unit = 'min' if 'min' in dur_str.lower() else ('Seasons' if 'season' in dur_str.lower() else 'Unknown')
    return val, unit

duration_parsed = cleaned.apply(parse_duration, axis=1)
cleaned['duration_num'] = [d[0] for d in duration_parsed]
cleaned['duration_type'] = [d[1] for d in duration_parsed]

# Age group demographic
def classify_demographic(rating):
    if rating in ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'G', 'TV-G']:
        return 'Kids & Family'
    elif rating in ['PG', 'TV-PG', 'PG-13', 'TV-14']:
        return 'Teens & Young Adults'
    elif rating in ['R', 'TV-MA', 'NC-17']:
        return 'Adults (18+)'
    else:
        return 'Unrated / Other'

cleaned['age_group'] = cleaned['rating_clean'].apply(classify_demographic)

# Content Freshness
cleaned['content_age_at_addition'] = (cleaned['year_added'] - cleaned['release_year']).apply(lambda x: max(0, x))
def classify_freshness(gap):
    if gap <= 0:
        return 'Direct-to-Netflix / Same Year'
    elif gap <= 2:
        return 'Recent Release (1-2 yrs)'
    elif gap <= 10:
        return 'Modern Catalog (3-10 yrs)'
    else:
        return 'Classic / Vintage (>10 yrs)'

cleaned['freshness_category'] = cleaned['content_age_at_addition'].apply(classify_freshness)
cleaned['primary_country'] = cleaned['country_clean'].apply(lambda x: x.split(',')[0].strip() if x != 'Unknown Country' else 'Unknown')
cleaned['primary_genre'] = cleaned['listed_in'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown')

cleaned[['title', 'type', 'release_year', 'year_added', 'duration_num', 'duration_type', 'age_group', 'freshness_category']].head(5)



## 5. Export Cleaned Data Artifacts


In [ ]:
out_csv = Path('../data/processed/netflix_cleaned.csv')
out_csv.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(out_csv, index=False)
print(f"[✓] Saved cleaned dataset to: {out_csv.resolve()}")
print(f"[✓] Final Dataset Shape: {cleaned.shape[0]} rows, {cleaned.shape[1]} columns")

